# 05 · Position angle and emission-line profiles

In slitless spectroscopy there is no slit to set the spectral resolution — an emission line is imaged as a *monochromatic picture of the galaxy*, dropped onto the detector at the line's wavelength. So the line's apparent **profile** is the galaxy's spatial extent **along the dispersion direction**. For an elongated (elliptical) galaxy that extent — and therefore the line width — depends on the galaxy's **position angle** relative to the dispersion.

This notebook makes that concrete: disperse one elongated galaxy with a single emission line at several orientations, extract the line, and watch its width change. This "morphological broadening" is a real systematic when measuring line widths or redshifts from grism spectra.

## 0 · Setup — an elongated galaxy and an emission-line spectrum

In [ ]:
import os
from pathlib import Path
os.environ.setdefault("JAX_COMPILATION_CACHE_DIR", str(Path.home() / ".cache" / "roman_grs_jax"))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import AsinhNorm
import jax
import jax.numpy as jnp

from roman_disperser import paths, psf_model, galaxy_disperser, sersic
from roman_disperser.elements import GRISM
from roman_disperser.optical_model import RomanOpticalModel
import roman_disperser.optical_model_jax as omj
import tutorial_helpers as th

SCA = 5
element = GRISM
model = RomanOpticalModel(config_file=str(paths.optical_model_path(element=element)))
opt = omj.make_sca_payload(model, sca=SCA, order="1")
psf = psf_model.get_or_make_psf_payload(detector=f"WFI{SCA:02d}", order="1", element=element,
                                        cache_dir=str(paths.psf_cache_dir()), verbose=False)
gal_disp = galaxy_disperser.make_galaxy_disperser(psf, opt)
OVERSAMPLE = psf["oversample"]
GAL_NPIX = 40 * OVERSAMPLE

wl_um, _, _ = th.wavelength_grid(element)
X_GAL, Y_GAL = 2000.0, 2000.0

# Emission-line-dominated spectrum: faint flat continuum + one narrow Gaussian
# line. The intrinsic line is much narrower than the morphological broadening we
# want to expose, so the extracted width is set by the galaxy shape, not the line.
LINE_UM, LINE_SIG_UM = 1.5, 0.0008          # ~8 Å intrinsic sigma
continuum = np.full_like(wl_um, 0.05)
line = 2.0 * np.exp(-0.5 * ((wl_um - LINE_UM) / LINE_SIG_UM) ** 2)
counts_in = jnp.asarray((continuum + line) * 50.0)

fig, ax = plt.subplots(figsize=(7, 2.6))
ax.plot(wl_um, np.asarray(counts_in), color="C0")
ax.set(xlabel="wavelength [µm]", ylabel="counts [e⁻/s]", title="input spectrum (one emission line)",
       xlim=(1.4, 1.6))
fig.tight_layout()

## 1 · One galaxy, several position angles

We use a single elongated exponential galaxy (axis ratio b/a = 0.3, r_eff = 0.5″) and rotate it. `theta` is the orientation on the SCA; at `theta = 90°` its major axis lies along the dispersion (y) direction.

In [ ]:
BA, REFF_ARCSEC, SERSIC_N = 0.3, 0.5, 1.0
reff_pix = sersic.catalog_r_eff_to_pixels(REFF_ARCSEC, 0.11, oversample=OVERSAMPLE)
thetas_deg = [0.0, 45.0, 90.0]

fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
stamps = {}
for ax, td in zip(axes, thetas_deg):
    g = np.asarray(sersic.make_sersic_image(reff_pix, SERSIC_N, BA, np.deg2rad(td), GAL_NPIX))
    stamps[td] = g
    ax.imshow(g, origin="lower", cmap="inferno",
              norm=AsinhNorm(linear_width=g.max()*0.02))
    ax.set(title=f"θ = {td:.0f}°", xticks=[], yticks=[])
fig.suptitle("the same galaxy at three position angles (dispersion is vertical)")
fig.tight_layout()

## 2 · Disperse each, and look at the emission line

The emission line lands at one point along the trace — its image there is the galaxy itself. With the major axis across the dispersion (θ = 0°) the line image is compact in y; rotated along the dispersion (θ = 90°) it is stretched in y, i.e. **smeared over more wavelengths**.

In [ ]:
# where does the line wavelength land on the trace?
lx, ly = th.spectral_trace(opt, X_GAL, Y_GAL, [LINE_UM])
lx, ly = int(round(float(lx[0]))), int(round(float(ly[0])))

images = {}
fig, axes = plt.subplots(1, 3, figsize=(9, 4))
for ax, td in zip(axes, thetas_deg):
    g = sersic.make_sersic_image(reff_pix, SERSIC_N, BA, np.deg2rad(td), GAL_NPIX)
    g = jnp.asarray(g / g.sum())
    out = np.asarray(gal_disp(g, X_GAL, Y_GAL, counts_in, jnp.asarray(wl_um),
                              jnp.zeros((4088, 4088), jnp.float32)))
    images[td] = out
    h = 70
    ax.imshow(out[ly-h:ly+h, lx-40:lx+40], origin="lower", cmap="inferno",
              norm=AsinhNorm(linear_width=out.max()*5.e-4, vmin=0, vmax=out.max()))
    ax.set(title=f"θ = {td:.0f}°", xticks=[], yticks=[])
fig.suptitle("emission-line image (zoom): elongates along dispersion as θ → 90°")
fig.tight_layout()

## 3 · Extract the line profile

Using the boxcar extractor from notebook 04, the recovered line profile broadens as the major axis swings toward the dispersion direction — a factor of ~2 in width here, set purely by the galaxy's orientation.

In [ ]:
def fwhm_angstrom(wl_um, prof):
    p = prof - np.median(prof)
    half = p.max() / 2.0
    idx = np.where(p >= half)[0]
    return (wl_um[idx[-1]] - wl_um[idx[0]]) * 1e4

fig, ax = plt.subplots(figsize=(7, 3.4))
for td, color in zip(thetas_deg, ["C0", "C1", "C3"]):
    prof = th.extract_1d(images[td], opt, X_GAL, Y_GAL, wl_um, aperture=25)
    fw = fwhm_angstrom(wl_um, prof)
    ax.plot(wl_um * 1e4, prof, color=color, label=f"θ = {td:.0f}°  (FWHM {fw:.0f} Å)")
ax.set(xlabel="wavelength [Å]", ylabel="count rate [e⁻/s per bin]",
       title="extracted emission line vs galaxy orientation", xlim=(14600, 15400))
ax.legend(); fig.tight_layout()
print(f"intrinsic line FWHM ≈ {2.355*LINE_SIG_UM*1e4:.0f} Å — the rest is morphological broadening")

## 4 · Why it matters

The line image is a *monochromatic image of the source*; its extent along the dispersion sets the observed line width. Consequences for slitless spectroscopy:

- a measured line width mixes the **intrinsic** width with the source's **morphology and orientation** — you cannot read kinematics off the width without modelling the morphology;
- the same galaxy observed at a different **roll** (notebook 03) gives a different line width and a shifted continuum, which is one reason multiple rolls are valuable;
- accurate redshifts/line fluxes need a model of the source shape — exactly the forward model `roman_disperser` provides.

## Recap

- An emission line in a grism is the galaxy's **monochromatic image** placed at the line wavelength.
- Its width along the dispersion — hence the extracted line profile — depends on the source **morphology and position angle** (morphological broadening), here ~2× across θ = 0–90°.

**Next — [06 · Catalogs and scaling](06_catalogs_scaling.ipynb).** Moving from hand-placed sources to catalog-driven fields, and the batching tricks that make large numbers of sources tractable.